# XGBoost Classification — F1 Per-Class Analysis

**Author:** Antoni Czolgowski  
**Course:** CSCI 5502 Data Mining — Spring 2026

In [1]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    ConfusionMatrixDisplay
)

import xgboost as xgb

from scipy.stats import pearsonr

RANDOM_STATE = 42
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

BASE = '../data/processed'

print('All imports loaded successfully.')

All imports loaded successfully.


## 1. Data Preparation

In [2]:
master_raw = pd.read_csv(f'{BASE}/master_dataset.csv', dtype={'county_fips': str})
master_scaled = pd.read_csv(f'{BASE}/master_dataset_scaled.csv', dtype={'county_fips': str})
volatility = pd.read_csv(f'{BASE}/county_volatility_dimTable.csv', dtype={'county_fips': str})

master_raw['county_fips'] = master_raw['county_fips'].str.zfill(5)
master_scaled['county_fips'] = master_scaled['county_fips'].str.zfill(5)
volatility['county_fips'] = volatility['county_fips'].str.zfill(5)

# Filter to 2024, merge volatility target & race_entropy_norm
df = master_raw[master_raw['election_year'] == 2024].copy().reset_index(drop=True)
entropy_2024 = master_scaled[master_scaled['election_year'] == 2024][['county_fips', 'race_entropy_norm']].copy()

df = df.merge(volatility[['county_fips', 'vol_z_abs_sum', 'swing_dir_score', 'vol_z_euc']],
              on='county_fips', how='left')
df = df.merge(entropy_2024, on='county_fips', how='left')

assert df.shape[0] == 250, f'Expected 250 rows, got {df.shape[0]}'
print(f'Merged dataset: {df.shape}')

# Create target variables
df['volatility_class'] = pd.qcut(
    df['vol_z_abs_sum'], q=5,
    labels=['Very Stable', 'Stable', 'Moderate', 'Volatile', 'Highly Volatile']
)
df['vol_quintile_num'] = pd.qcut(df['vol_z_abs_sum'], q=5, labels=False) + 1
df['vol_binary'] = (df['vol_quintile_num'] >= 4).astype(int)

print('Class distribution:')
print(df['volatility_class'].value_counts().sort_index())

Merged dataset: (250, 43)
Class distribution:
volatility_class
Very Stable        50
Stable             50
Moderate           50
Volatile           50
Highly Volatile    50
Name: count, dtype: int64


In [3]:
# Feature selection
exclude_cols = [
    'county_fips', 'state', 'county_name', 'election_year',
    'dem_votes', 'rep_votes', 'total_votes', 'dem_pct', 'rep_pct', 'dem_margin',
    'vol_z_abs_sum', 'vol_z_euc', 'swing_dir_score',
    'volatility_class', 'vol_quintile_num', 'vol_binary'
]

feature_cols = [c for c in df.columns if c not in exclude_cols]
df_features = df[feature_cols].copy()

# Log-transform skewed variables
log_cols = ['total_population', 'population_density',
            'median_household_income', 'median_home_value', 'median_gross_rent']

for col in log_cols:
    df_features[f'log_{col}'] = np.log1p(df_features[col])
df_features.drop(columns=log_cols, inplace=True)

# Multicollinearity check & drop
drop_features = ['pct_income_over_100k']
if 'pct_non_hispanic_white' in df_features.columns and 'race_entropy_norm' in df_features.columns:
    r_val, _ = pearsonr(df_features['pct_non_hispanic_white'], df_features['race_entropy_norm'])
    if abs(r_val) > 0.85:
        drop_features.append('pct_non_hispanic_white')
df_features.drop(columns=drop_features, inplace=True)

feature_names = list(df_features.columns)

# StandardScale
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df_features), columns=feature_names, index=df_features.index)

y = df['vol_quintile_num'].values
y_binary = df['vol_binary'].values
CLASS_NAMES = ['Q1_VeryStable', 'Q2_Stable', 'Q3_Moderate', 'Q4_Volatile', 'Q5_HighlyVolatile']

print(f'X shape: {X.shape}, y classes: {np.unique(y)}')
print(f'Features: {feature_names}')

X shape: (250, 28), y classes: [1 2 3 4 5]
Features: ['median_age', 'pct_black', 'pct_asian', 'pct_two_or_more_races', 'pct_hispanic', 'pct_hs_or_higher', 'pct_bachelors_plus', 'pct_below_poverty', 'pct_income_under_25k', 'pct_income_50k_100k', 'unemployment_rate', 'pct_owner_occupied', 'pct_drive_alone', 'pct_carpool', 'pct_public_transit', 'pct_work_from_home', 'pct_family_households', 'pct_married_couple', 'pct_living_alone', 'pct_senior_65plus', 'pct_young_adult_18_24', 'pct_foreign_born', 'race_entropy_norm', 'log_total_population', 'log_population_density', 'log_median_household_income', 'log_median_home_value', 'log_median_gross_rent']


## 2. Helper Functions

In [4]:
def nested_cv(X, y, base_model, param_grid, n_iter=80, model_name='Model'):
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    n_classes = len(np.unique(y))
    all_preds = np.zeros(len(y), dtype=int)
    all_proba = np.zeros((len(y), n_classes))
    fold_metrics = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_train = X.iloc[train_idx] if hasattr(X, 'iloc') else X[train_idx]
        X_test = X.iloc[test_idx] if hasattr(X, 'iloc') else X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=param_grid,
            n_iter=min(n_iter, np.prod([len(v) if isinstance(v, list) else 1 for v in param_grid.values()])),
            cv=inner_cv,
            scoring='f1_macro',
            random_state=RANDOM_STATE,
            n_jobs=-1,
            error_score='raise'
        )
        search.fit(X_train, y_train)

        preds = search.best_estimator_.predict(X_test)
        proba = search.best_estimator_.predict_proba(X_test)

        all_preds[test_idx] = preds
        all_proba[test_idx] = proba

        fold_f1 = f1_score(y_test, preds, average='macro')
        fold_acc = accuracy_score(y_test, preds)
        fold_metrics.append({
            'fold': fold_idx,
            'accuracy': fold_acc,
            'f1_macro': fold_f1,
            'best_params': search.best_params_
        })
        print(f'  Fold {fold_idx+1}: acc={fold_acc:.3f}, F1-macro={fold_f1:.3f}')

    return all_preds, all_proba, fold_metrics


def evaluate_model(y_true, y_pred, y_proba, y_binary, class_names, model_name='Model'):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro')
    rec = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    f1_per_class = f1_score(y_true, y_pred, average=None)

    proba_high = y_proba[:, 3] + y_proba[:, 4] if y_proba.shape[1] == 5 else y_proba[:, 1]
    auc_binary = roc_auc_score(y_binary, proba_high)

    print(f'\n=== {model_name} — Evaluation ===')
    print(f'Accuracy:        {acc:.4f}')
    print(f'Precision (macro): {prec:.4f}')
    print(f'Recall (macro):    {rec:.4f}')
    print(f'F1-macro:          {f1:.4f}')
    print(f'ROC-AUC (binary):  {auc_binary:.4f}')
    print(f'\nPer-class F1: {dict(zip(class_names, np.round(f1_per_class, 4)))}')
    print(f'\n{classification_report(y_true, y_pred, target_names=class_names)}')

    return {'accuracy': acc, 'precision_macro': prec, 'recall_macro': rec,
            'f1_macro': f1, 'roc_auc_binary': auc_binary, 'f1_per_class': f1_per_class}

## 3. XGBoost — Nested CV + Evaluation

In [5]:
y_xgb = y - 1

xgb_base = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    verbosity=0
)

xgb_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [1.0, 2.0, 5.0]
}

print('XGBoost — Nested CV (5-fold outer, 3-fold inner, 100 iter):')
xgb_preds_0idx, xgb_proba, xgb_fold_metrics = nested_cv(
    X, y_xgb, xgb_base, xgb_param_grid, n_iter=100, model_name='XGBoost'
)

xgb_preds = xgb_preds_0idx + 1

print('\nBest params per fold:')
for fm in xgb_fold_metrics:
    print(f'  Fold {fm["fold"]+1}: {fm["best_params"]}')

XGBoost — Nested CV (5-fold outer, 3-fold inner, 100 iter):
  Fold 1: acc=0.320, F1-macro=0.333
  Fold 2: acc=0.280, F1-macro=0.259
  Fold 3: acc=0.380, F1-macro=0.380
  Fold 4: acc=0.460, F1-macro=0.426
  Fold 5: acc=0.480, F1-macro=0.476

Best params per fold:
  Fold 1: {'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 10, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
  Fold 2: {'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0, 'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
  Fold 3: {'subsample': 1.0, 'reg_lambda': 5.0, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
  Fold 4: {'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
  Fold 5: {'subsample': 0.7, 'reg_lambda'

In [6]:
xgb_metrics = evaluate_model(y, xgb_preds, xgb_proba, y_binary, CLASS_NAMES, 'XGBoost')


=== XGBoost — Evaluation ===
Accuracy:        0.3840
Precision (macro): 0.3826
Recall (macro):    0.3840
F1-macro:          0.3826
ROC-AUC (binary):  0.7780

Per-class F1: {'Q1_VeryStable': np.float64(0.3673), 'Q2_Stable': np.float64(0.2737), 'Q3_Moderate': np.float64(0.3265), 'Q4_Volatile': np.float64(0.3853), 'Q5_HighlyVolatile': np.float64(0.56)}

                   precision    recall  f1-score   support

    Q1_VeryStable       0.38      0.36      0.37        50
        Q2_Stable       0.29      0.26      0.27        50
      Q3_Moderate       0.33      0.32      0.33        50
      Q4_Volatile       0.36      0.42      0.39        50
Q5_HighlyVolatile       0.56      0.56      0.56        50

         accuracy                           0.38       250
        macro avg       0.38      0.38      0.38       250
     weighted avg       0.38      0.38      0.38       250



## 4. SVM — Nested CV + Evaluation

In [7]:
from sklearn.svm import SVC

# Linear kernel
svm_linear_base = SVC(kernel='linear', probability=True, random_state=RANDOM_STATE)

svm_linear_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': ['balanced', None]
}

print('SVM (Linear) — Nested CV:')
svm_lin_preds, svm_lin_proba, svm_lin_fold_metrics = nested_cv(
    X, y, svm_linear_base, svm_linear_grid, n_iter=10, model_name='SVM-Linear'
)

svm_lin_metrics = evaluate_model(y, svm_lin_preds, svm_lin_proba, y_binary, CLASS_NAMES, 'SVM (Linear)')

SVM (Linear) — Nested CV:
  Fold 1: acc=0.360, F1-macro=0.359
  Fold 2: acc=0.280, F1-macro=0.285
  Fold 3: acc=0.320, F1-macro=0.317
  Fold 4: acc=0.420, F1-macro=0.398
  Fold 5: acc=0.460, F1-macro=0.447

=== SVM (Linear) — Evaluation ===
Accuracy:        0.3680
Precision (macro): 0.3647
Recall (macro):    0.3680
F1-macro:          0.3641
ROC-AUC (binary):  0.7651

Per-class F1: {'Q1_VeryStable': np.float64(0.4561), 'Q2_Stable': np.float64(0.3654), 'Q3_Moderate': np.float64(0.1915), 'Q4_Volatile': np.float64(0.3441), 'Q5_HighlyVolatile': np.float64(0.4632)}

                   precision    recall  f1-score   support

    Q1_VeryStable       0.41      0.52      0.46        50
        Q2_Stable       0.35      0.38      0.37        50
      Q3_Moderate       0.20      0.18      0.19        50
      Q4_Volatile       0.37      0.32      0.34        50
Q5_HighlyVolatile       0.49      0.44      0.46        50

         accuracy                           0.37       250
        macro avg 

In [8]:
# RBF kernel
svm_rbf_base = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)

svm_rbf_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'class_weight': ['balanced', None]
}

print('SVM (RBF) — Nested CV:')
svm_rbf_preds, svm_rbf_proba, svm_rbf_fold_metrics = nested_cv(
    X, y, svm_rbf_base, svm_rbf_grid, n_iter=30, model_name='SVM-RBF'
)

svm_rbf_metrics = evaluate_model(y, svm_rbf_preds, svm_rbf_proba, y_binary, CLASS_NAMES, 'SVM (RBF)')

SVM (RBF) — Nested CV:
  Fold 1: acc=0.320, F1-macro=0.321
  Fold 2: acc=0.300, F1-macro=0.301
  Fold 3: acc=0.400, F1-macro=0.396
  Fold 4: acc=0.360, F1-macro=0.339
  Fold 5: acc=0.380, F1-macro=0.364

=== SVM (RBF) — Evaluation ===
Accuracy:        0.3520
Precision (macro): 0.3456
Recall (macro):    0.3520
F1-macro:          0.3468
ROC-AUC (binary):  0.7899

Per-class F1: {'Q1_VeryStable': np.float64(0.4545), 'Q2_Stable': np.float64(0.2264), 'Q3_Moderate': np.float64(0.3168), 'Q4_Volatile': np.float64(0.1647), 'Q5_HighlyVolatile': np.float64(0.5714)}

                   precision    recall  f1-score   support

    Q1_VeryStable       0.42      0.50      0.45        50
        Q2_Stable       0.21      0.24      0.23        50
      Q3_Moderate       0.31      0.32      0.32        50
      Q4_Volatile       0.20      0.14      0.16        50
Q5_HighlyVolatile       0.58      0.56      0.57        50

         accuracy                           0.35       250
        macro avg       

In [9]:
# Select best kernel
if svm_rbf_metrics['f1_macro'] >= svm_lin_metrics['f1_macro']:
    svm_best_kernel = 'RBF'
    svm_preds = svm_rbf_preds
    svm_metrics = svm_rbf_metrics
    print('-> RBF kernel selected (higher F1-macro)')
else:
    svm_best_kernel = 'Linear'
    svm_preds = svm_lin_preds
    svm_metrics = svm_lin_metrics
    print('-> Linear kernel selected (higher F1-macro)')

-> Linear kernel selected (higher F1-macro)


## 5. F1 Comparison Tables

In [10]:
display_names = ['Highly Stable', 'Stable', 'Moderate', 'Volatile', 'Highly Volatile']

# --- XGBoost F1 Table ---
xgb_f1_df = pd.DataFrame({
    'Volatility Class': display_names,
    'F1': [f'{v:.4f}' for v in xgb_metrics['f1_per_class']]
}).set_index('Volatility Class').T

print('XGBoost: F1 Comparison')
print(xgb_f1_df.to_string())
print('\nTable 11: XGBoost F1 score distribution across volatility classes')

print('\n' + '='*80 + '\n')

# --- SVM F1 Table ---
svm_f1_df = pd.DataFrame({
    'Volatility Class': display_names,
    'F1': [f'{v:.4f}' for v in svm_metrics['f1_per_class']]
}).set_index('Volatility Class').T

print(f'SVM ({svm_best_kernel}): F1 Comparison')
print(svm_f1_df.to_string())
print(f'\nTable 12: SVM ({svm_best_kernel}) F1 score distribution across volatility classes')

XGBoost: F1 Comparison
Volatility Class Highly Stable  Stable Moderate Volatile Highly Volatile
F1                      0.3673  0.2737   0.3265   0.3853          0.5600

Table 11: XGBoost F1 score distribution across volatility classes


SVM (Linear): F1 Comparison
Volatility Class Highly Stable  Stable Moderate Volatile Highly Volatile
F1                      0.4561  0.3654   0.1915   0.3441          0.4632

Table 12: SVM (Linear) F1 score distribution across volatility classes


In [11]:
# Styled HTML tables for nicer display
from IPython.display import display, HTML

# XGBoost table
xgb_html_df = pd.DataFrame({
    'Volatility Class': display_names,
    'F1': [f'{v:.4f}' for v in xgb_metrics['f1_per_class']]
}).set_index('Volatility Class').T

styled_xgb = xgb_html_df.style.set_caption(
    'Table 11: XGBoost F1 score distribution across volatility classes'
).set_table_styles([
    {'selector': 'caption', 'props': [('font-size', '12px'), ('font-style', 'italic'), ('color', '#666')]},
    {'selector': 'th', 'props': [('background-color', '#d4a843'), ('color', 'white'), ('padding', '6px 12px')]},
    {'selector': 'td', 'props': [('padding', '6px 12px'), ('text-align', 'center')]}
])

print('XGBoost: F1 Comparison')
display(styled_xgb)

print()

# SVM table
svm_html_df = pd.DataFrame({
    'Volatility Class': display_names,
    'F1': [f'{v:.4f}' for v in svm_metrics['f1_per_class']]
}).set_index('Volatility Class').T

styled_svm = svm_html_df.style.set_caption(
    f'Table 12: SVM ({svm_best_kernel}) F1 score distribution across volatility classes'
).set_table_styles([
    {'selector': 'caption', 'props': [('font-size', '12px'), ('font-style', 'italic'), ('color', '#666')]},
    {'selector': 'th', 'props': [('background-color', '#d4a843'), ('color', 'white'), ('padding', '6px 12px')]},
    {'selector': 'td', 'props': [('padding', '6px 12px'), ('text-align', 'center')]}
])

print(f'SVM ({svm_best_kernel}): F1 Comparison')
display(styled_svm)

XGBoost: F1 Comparison


Volatility Class,Highly Stable,Stable,Moderate,Volatile,Highly Volatile
F1,0.3673,0.2737,0.3265,0.3853,0.5600



SVM (Linear): F1 Comparison


Volatility Class,Highly Stable,Stable,Moderate,Volatile,Highly Volatile
F1,0.4561,0.3654,0.1915,0.3441,0.4632
